## Hydrodynamic model of the Southern North Sea

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mikeio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import modelskill as ms
import torch
from sklearn.manifold import TSNE

# dataset helper
from helper import get_wl_point_obs, plot_shred_reconstruction, extract_water_level_measurements, align_ds

import sys
from pathlib import Path

# Add parent directory to path to access src module
sys.path.insert(0, str(Path.cwd().parent.parent))

# Importing modules from src
from src.datamanager import ROMDataManager
from src.engine import SHREDEngine
from src.train import fit, evaluate
from src.train import lagged_fit, lagged_evaluate
from src.mSHRED.mSHRED import mSHRED
from src.utils import plot_training_curves, evaluate_model

np.random.seed(42)

# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ModuleNotFoundError: No module named 'helper'

## Load Data and Measurement

In [4]:
# folders
dfs_fldr = "datasets/output/Area.dfsu"
msh_fldr = "datasets/input/Mesh_8000_elems_HD_v2.mesh"
obs_fldr  = "datasets/observations/"

msh = mikeio.open(msh_fldr)
dfs = mikeio.open(dfs_fldr)
geom = dfs.geometry

# read datasets
ds = dfs.read()
dsH = ds["Surface elevation"]
dsU = ds["U velocity"]
dsV = ds["V velocity"]

del dfs

FileNotFoundError: File datasets/input/Mesh_8000_elems_HD_v2.mesh not found

In [3]:
# sensor coverage
alllist = get_wl_point_obs()

# For sparse data simulation
hf = ["Dover", "ZeebruggeWielingendok", "NWA", "K141"]
alllist = [s for s in alllist if s.name in hf]

stations = []
available_stations = []
intermittent = []
for i in range(len(alllist)):
    stations.append(alllist[i])
    if ds.time.isin(alllist[i].time).all():
        available_stations.append(alllist[i])
    else:
        intermittent.append(alllist[i])

# extract water level measurements for all stations
measurements = extract_water_level_measurements(stations, ds_time=ds.time, obs_fldr=obs_fldr)

fig, axs = plt.subplots(1,3, figsize = (12,4))
ms.plotting.spatial_overview(stations, mod = msh.geometry, figsize = (5,5), ax=axs[0], title=f"Stations: {len(stations)}")
ms.plotting.spatial_overview(available_stations, mod = msh.geometry, figsize = (5,5), ax=axs[1], title=f"Always available stations: {len(available_stations)}/{len(stations)}")
ms.plotting.spatial_overview(intermittent, mod = msh.geometry, figsize = (5,5), ax=axs[2], title=f"Intermittent stations: {len(intermittent)}/{len(stations)}")
plt.tight_layout()
plt.show()


NameError: name 'get_wl_point_obs' is not defined

In [ ]:
# remove nans from H, U, V 
dsH = dsH.values[:,~np.isnan(ds["Surface elevation"].values).any(axis=0)]
dsU = dsU.values[:,~np.isnan(ds["Surface elevation"].values).any(axis=0)]
dsV = dsV.values[:,~np.isnan(ds["Surface elevation"].values).any(axis=0)]

# outputs
H = torch.tensor(dsH, dtype=torch.float32)
U = torch.tensor(dsU, dtype=torch.float32)
V = torch.tensor(dsV, dtype=torch.float32)

# inputs
measurements = torch.tensor(measurements, dtype=torch.float32)

# Extract measurements for always-available stations only
av_indices = [stations.index(s) for s in available_stations]
measurements_av = measurements[:, av_indices]

# print the shape of the arrays
print("H shape:", H.shape)
print("U shape:", U.shape)
print("V shape:", V.shape)
print("Measurements shape:", measurements.shape)
print("Always-available measurements shape:", measurements_av.shape)

# free memory
del dsH, dsU, dsV
del ds

## ROM training

In [ ]:
# Assuming 30 days per month, 24 hours per day, and measurements every 30 minutes (2 per hour)
lenmonths = (30 * 24 * 2)  
nmonths = H.shape[0] // lenmonths 
ntime_keeped = nmonths * lenmonths 
print(f"Time keeprd: {ntime_keeped}/{H.shape[0]} total steps ({nmonths} months) -> coverage: {100*ntime_keeped/H.shape[0]:.1f}% \n")

# reshape the arrays to (nmonths, lenmonths, nfeatures) for training
H = H[:ntime_keeped].reshape(nmonths, lenmonths, -1)
U = U[:ntime_keeped].reshape(nmonths, lenmonths, -1)
V = V[:ntime_keeped].reshape(nmonths, lenmonths, -1)
measurements = measurements[:ntime_keeped].reshape(nmonths, lenmonths, -1)
measurements_av = measurements_av[:ntime_keeped].reshape(nmonths, lenmonths, -1)    

print("Reshaped H shape:", H.shape)
print("Reshaped U shape:", U.shape)
print("Reshaped V shape:", V.shape)
print("Reshaped measurements shape:", measurements.shape)
print("Reshaped measurements shape:", measurements_av.shape)

#permute the months 
permuted_indices = np.random.permutation(nmonths)
H = H[permuted_indices]
U = U[permuted_indices]
V = V[permuted_indices]
measurements = measurements[permuted_indices]
measurements_av = measurements_av[permuted_indices]

print("\npermutation indices:", permuted_indices)

## Hyperparameters and tuning

In [ ]:
ERRORS = []

# DataManager
LAGS = 48               # number of past time steps to use as input for the model (e.g., 48 for 24 hours of data at 30-minute intervals)

TRAIN_SIZE = 0.8        # percentage of data used for training 
VAL_SIZE = 0.1          # percentage of data used for validation
TEST_SIZE = 0.1         # percentage of data used for testing

COMPRESS_H = 12         # modes kept for H (surface elevation)
COMPRESS_U = 40         # modes kept for U (U velocity)
COMPRESS_V = 68         # modes kept for V (V velocity)

# Training
OPTIMIZER = "AdamW"     # optimizer to use
LEARNING_RATE = 1e-3    # learning rate for training
NUM_EPOCHS = 150        # maximum number of epochs for training
LR_SCHEDULER = "CosineAnnealingLR"      # learning rate scheduler to use
SCHEDULER_STEP_SIZE = NUM_EPOCHS        # step size for learning rate scheduler (if applicable)

BATCH_SIZE = 32         # batch size for training
PATIENCE = 30           # number of epochs with no improvement on validation loss before stopping training
VERBOSE_TRAIN = True

# Default configuration used by the training cells below
W_FACTOR = 0.8              # weight factor for the loss function (if using a weighted loss)
W_KEEPED_TIMESTEPS = 16      # number of past time steps to keep in the loss function when using a weighted loss (if applicable)

# Architecture
ENCODER_DEPTH = 2              # number of encoder layers
HIDDEN_SIZE = 32                # hidden size for encoder and decoder layers
N_HEADS = 4                     # number of attention heads (if using a transformer-based architecture)
DIM_FEEDFORWARD = 48            # dimension of feedforward network in transformer (if using a transformer-based architecture)
DECODER = "cnn"                 # type of decoder to use (mpl or cnn)   
DECODER_HIDDEN_SIZES = [64]     # hidden sizes for decoder layers 
DROPOUT = 0.0                   # dropout rate for encoder and decoder layers
MASKING_TYPE = "mask_data"      # type of masking to apply to inputs during training (mask_data, mask_data_random, etc.)

In [ ]:
# Lagged-loss tuning grid
W_FACTOR_GRID = np.round(np.arange(0.2, 0.9, 0.1), 2).tolist()  # [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
W_KEEPED_TIMESTEPS_GRID = np.arange(4, 21, 4).tolist()  # [4, 8, 12, 16, 20]

LAGGED_LOSS_TUNING_GRID = [
    {"w_factor": w_factor, "w_keeped_timesteps": w_keeped_timesteps}
    for w_factor in W_FACTOR_GRID
    for w_keeped_timesteps in W_KEEPED_TIMESTEPS_GRID
]

print("Lagged-loss tuning grid, total configurations:", len(LAGGED_LOSS_TUNING_GRID))
print("Expected total times (15 min per config) -> total expected time: ", len(LAGGED_LOSS_TUNING_GRID)*15, "minutes (", len(LAGGED_LOSS_TUNING_GRID)*15/60, "hours )\n")

## Normal training

In [ ]:
# Data manager
manager = ROMDataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id = "Surface elevation" , compress = COMPRESS_H, measurements = measurements)
manager.add_data(U, id = "U velocity", compress = COMPRESS_U)
manager.add_data(V, id = "V velocity", compress = COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose = True, masking_type=MASKING_TYPE)

# network input and output shapes
nn_input_shape = train_dataset.X.shape[-1]
nn_output_shape = train_dataset.Y.shape[-1]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels = True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

# check if the model is already trained and load the weights if available
model_path = Path("models/mSHRED_HYDRO_NORMAL.pth")
if model_path.exists():
    print(f"Loading pre-trained model from {model_path}")
    mshred.load_state_dict(torch.load(model_path, map_location=device))

else:

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE, masking_type=MASKING_TYPE,
    )

    # Save the trained model
    model_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(mshred.state_dict(), model_path)

    # PLOT TRAINING CURVES
    plot_training_curves(train_errors, valid_errors)

    # EVALUATION
    test_error = evaluate_model(
        evaluate, mshred, train_dataset, val_dataset, test_dataset, 
        model = f"(mSHRED): TR+{DECODER}", type="Normal_training", masking_type=MASKING_TYPE,
    )

    ERRORS.append(test_error)
    dataframe_errors = pd.DataFrame(ERRORS)
    display(dataframe_errors)
    dataframe_errors.to_csv("dataframe_errors.csv", index=False)

normal_shred = mshred

In [ ]:
# mSHRED RECONSTRUCTION
# engine = SHREDEngine(manager, mshred)
# test_reconstruction = engine.sensor_to_reconstruction_dataset(test_dataset)
# ds_r = align_ds(manager.test_indices, ds, permuted_indices, lenmonths, geom)
# ds_shred, ds_error = plot_shred_reconstruction(test_reconstruction, ds_r, geom, title_suffix="Normal_training")

## Lagged training

In [ ]:
# Data manager
manager = ROMDataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id = "Surface elevation" , compress = COMPRESS_H, measurements = measurements)
manager.add_data(U, id = "U velocity", compress = COMPRESS_U)
manager.add_data(V, id = "V velocity", compress = COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose = True, pad_output=True, masking_type=MASKING_TYPE)

# network input and output shapes
nn_input_shape = train_dataset.X.shape[-1]
nn_output_shape = train_dataset.Y.shape[-1] 
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels = True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, decoder_3D = True
)
mshred = mSHRED(args=model_args).to(device)

In [ ]:
# check if the model is already trained and load the weights if available
model_path = Path("models/mSHRED_HYDRO_LAGGED.pth")
if model_path.exists():
    print(f"Loading pre-trained model from {model_path}")
    mshred.load_state_dict(torch.load(model_path, map_location=device))

else:

    # Use cosine annealing LR schedule
    train_errors, valid_errors = lagged_fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE, w_factor=W_FACTOR, w_keeped_steps=W_KEEPED_TIMESTEPS, masking_type=MASKING_TYPE,
    )

    # Save the trained model
    model_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(mshred.state_dict(), model_path)

    # PLOT TRAINING CURVES
    plot_training_curves(train_errors, valid_errors)

    # EVALUATION
    test_error = evaluate_model(
        lagged_evaluate, mshred, train_dataset, val_dataset, test_dataset, 
        model = f"(mSHRED): TR+{DECODER}", type="Lagged_training", masking_type=MASKING_TYPE,
    )

    ERRORS.append(test_error)
    dataframe_errors = pd.DataFrame(ERRORS)
    display(dataframe_errors)
    dataframe_errors.to_csv("dataframe_errors.csv", index=False)

lagged_shred = mshred

In [ ]:
# mSHRED RECONSTRUCTION
#engine = SHREDEngine(manager, mshred)
#test_reconstruction = engine.sensor_to_reconstruction_dataset_lagged(test_dataset)
#ds_r = align_ds(manager.test_indices, ds, permuted_indices, lenmonths, geom)
#ds_shred, ds_error = plot_shred_reconstruction(test_reconstruction, ds_r, geom, title_suffix="Lagged_training")

#del engine, test_reconstruction, ds_shred, ds_error, ds_r, mshred, model_args, train_errors, valid_errors, test_error

## Embedding comparison

In [ ]:
def get_embeddings(model, dataset, device, n_samples=200):
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=n_samples, shuffle=False)

    if MASKING_TYPE in ["mask_data", "mask_data_random"]:
        x, m, _ = next(iter(loader))
        x = x.to(device)
        m = m.to(device)

        print("Input batch shape:", x.shape)  # (n_samples, lags, input_size)

        with torch.no_grad():
            H = model.encoder(x,m)

    else:
        x, _ = next(iter(loader))
        x = x.to(device)

        print("Input batch shape:", x.shape)  # (n_samples, lags, input_size)

        with torch.no_grad():
            H = model.encoder(x)

    return H.cpu().numpy()

def plot_temporal_embeddings(H_normal, H_lagged, n_samples=5, n_dims=4, supervision_t=W_KEEPED_TIMESTEPS):
    lags = H_normal.shape[1]

    fig, axes = plt.subplots(n_samples, n_dims, figsize=(4.5*n_dims, 4.5*n_samples))

    for i in range(n_samples):
        idx = np.random.randint(H_normal.shape[0])

        for d in range(n_dims):
            ax = axes[i, d] if n_samples > 1 else axes[d]

            ax.plot(range(lags), H_normal[idx, :, d], label='Normal', linewidth=2)
            ax.plot(range(lags), H_lagged[idx, :, d], label='Lagged', linewidth=2)

            # ── supervision marker ─────────────────────────
            if supervision_t is not None:
                ax.axvline(
                    x=LAGS-supervision_t,
                    color='black',
                    linestyle='--',
                    linewidth=1.5,
                    label='Supervision' 
                )

            ax.set_title(f'Sample {idx} - Latent dim {d}')
            ax.set_xlabel('Time lag')
            ax.set_ylabel('Centered activation')
            ax.legend()

    plt.tight_layout()
    plt.show()

def plot_temporal_embeddings_mean(H_normal, H_lagged, n_dims=4, supervision_t=W_KEEPED_TIMESTEPS):
    
    mean_normal = H_normal.mean(axis=0)
    mean_lagged = H_lagged.mean(axis=0)
    std_normal = H_normal.std(axis=0)
    std_lagged = H_lagged.std(axis=0)
    lags = mean_normal.shape[0]
    n_dims = min(n_dims, mean_normal.shape[1])

    fig, axes = plt.subplots(1, n_dims, figsize=(5 * n_dims, 5), sharey=True)
    if n_dims == 1:
        axes = [axes]

    for d in range(n_dims):
        ax = axes[d]
        ax.plot(range(lags), mean_normal[:, d], label='Normal', linewidth=2)
        ax.fill_between(
            range(lags),
            mean_normal[:, d] - std_normal[:, d],
            mean_normal[:, d] + std_normal[:, d],
            alpha=0.2,
            label='Normal ±1 std',
        )
        ax.plot(range(lags), mean_lagged[:, d], label='Lagged', linewidth=2)
        ax.fill_between(
            range(lags),
            mean_lagged[:, d] - std_lagged[:, d],
            mean_lagged[:, d] + std_lagged[:, d],
            alpha=0.2,
            label='Lagged ±1 std',
        )

        if supervision_t is not None:
            ax.axvline(
                x=LAGS - supervision_t,
                color='black',
                linestyle='--',
                linewidth=1.5,
                label='Supervision'
            )

        ax.set_title(f'Mean latent dim {d}')
        ax.set_xlabel('Time lag')
        if d == 0:
            ax.set_ylabel('Centered activation')
        ax.legend()

    plt.tight_layout()
    plt.show()

def total_variation(H):
    # differences along time axis: (N, T-1, D)
    diff = H[:, 1:, :] - H[:, :-1, :]

    # L2 norm over latent dimension -> (N, T-1)
    step_norms = np.linalg.norm(diff, axis=2)

    # sum over time -> (N,)
    tv_per_sample = step_norms.sum(axis=1)

    return tv_per_sample

def compute_tv_stats(H_normal, H_lagged):
    tv_normal = total_variation(H_normal)
    tv_lagged = total_variation(H_lagged)

    stats = {
        "normal_mean": tv_normal.mean(),
        "normal_std": tv_normal.std(),
        "lagged_mean": tv_lagged.mean(),
        "lagged_std": tv_lagged.std(),
    }

    return stats, tv_normal, tv_lagged

In [ ]:
H_normal = get_embeddings(normal_shred, test_dataset, device)
H_lagged = get_embeddings(lagged_shred, test_dataset, device)

print("H_normal shape:", H_normal.shape)  # (lags, n_samples, hidden_size)
print("H_lagged shape:", H_lagged.shape)  # (lags, n_samples, hidden_size)

H_normal = np.transpose(H_normal, (1, 0, 2))
H_lagged = np.transpose(H_lagged, (1, 0, 2))

print("H_normal permuted shape:", H_normal.shape)  # (n_samples, lags, hidden_size)
print("H_lagged permuted shape:", H_lagged.shape)  # (n_samples, lags, hidden_size)

H_normal_centered = H_normal - H_normal.mean(axis=1, keepdims=True)
H_lagged_centered = H_lagged - H_lagged.mean(axis=1, keepdims=True)

plot_temporal_embeddings(H_normal_centered, H_lagged_centered)

In [ ]:
plot_temporal_embeddings_mean(H_normal_centered, H_lagged_centered)

In [ ]:
stats, tv_normal, tv_lagged = compute_tv_stats(H_normal, H_lagged)

print("Normal  TV:  ", stats["normal_mean"], "±", stats["normal_std"])
print("Lagged  TV:  ", stats["lagged_mean"], "±", stats["lagged_std"])

## New plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ------------------------------------------------------------------
# Style (same as the error-accumulation figure)
# ------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "legend.frameon": False,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

C_NORMAL = "#1f77b4"   # same blue as the baseline
C_LAGGED = "#ff7f0e"   # same orange as the proposed method

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

def total_variation(H, normalize=True, eps=1e-12):
    """H: (N, T, D). Optionally scale-normalize each sample by its RMS."""
    if normalize:
        H = H / (H.std(axis=(1, 2), keepdims=True) + eps)
    diff = H[:, 1:, :] - H[:, :-1, :]
    return np.linalg.norm(diff, axis=2).sum(axis=1)

def compute_tv_stats(H_normal, H_lagged, normalize=True):
    tv_n = total_variation(H_normal, normalize)
    tv_l = total_variation(H_lagged, normalize)
    stats = {
        "normal_mean": tv_n.mean(), "normal_std": tv_n.std(),
        "lagged_mean": tv_l.mean(), "lagged_std": tv_l.std(),
        "change_pct": 100 * (tv_l.mean() - tv_n.mean()) / tv_n.mean(),
    }
    return stats, tv_n, tv_l


def plot_temporal_embeddings_unified(
    H_normal, H_lagged,
    n_dims=4, n_samples=2,
    supervision_t=None,            # W: number of supervised steps
    label_normal="T-SHRED",
    label_lagged="Lagged T-SHRED",
    sample_idx=None, seed=0,
    show_tv=True, tv_normalize=True,
    savepath=None,
):
    N, lags, hidden = H_normal.shape
    n_dims = min(n_dims, hidden)
    t = np.arange(lags)

    rng = np.random.default_rng(seed)
    if sample_idx is None:
        sample_idx = rng.choice(N, size=n_samples, replace=False)
    n_samples = len(sample_idx)
    n_rows = n_samples + 1

        # ---- TV stats go into the legend labels ----
    lab_n, lab_l = label_normal, label_lagged
    bold_idx = None                      # indice dell'handle da mettere in grassetto
    if show_tv:
        s, _, _ = compute_tv_stats(H_normal, H_lagged, tv_normalize)
        lab_n = f"{label_normal}  (TV = {s['normal_mean']:.1f} ± {s['normal_std']:.1f})"
        lab_l = f"{label_lagged}  (TV = {s['lagged_mean']:.1f} ± {s['lagged_std']:.1f})"
        bold_idx = 0 if s["normal_mean"] < s["lagged_mean"] else 1
        print(f"TV change (lagged vs normal): {s['change_pct']:+.1f}%")

    fig, axes = plt.subplots(
        n_rows, n_dims, figsize=(4.2 * n_dims, 2.8 * n_rows),
        sharex=True, constrained_layout=True, squeeze=False,
    )
    row_labels = [f"Sample {i}" for i in sample_idx] + ["Mean ± 1 std"]

    for d in range(n_dims):
        for r, i in enumerate(sample_idx):
            ax = axes[r, d]
            ax.plot(t, H_normal[i, :, d], color=C_NORMAL, lw=1.8)
            ax.plot(t, H_lagged[i, :, d], color=C_LAGGED, lw=1.8)

        ax = axes[-1, d]
        for H, c in [(H_normal, C_NORMAL), (H_lagged, C_LAGGED)]:
            mu, sd = H[:, :, d].mean(0), H[:, :, d].std(0)
            ax.plot(t, mu, color=c, lw=2.2)
            ax.fill_between(t, mu - sd, mu + sd, color=c, alpha=0.2, lw=0)

        for r in range(n_rows):
            ax = axes[r, d]
            if supervision_t is not None:
                x0 = lags - supervision_t
                ax.axvspan(x0, lags - 1, color="0.2", alpha=0.10, lw=0, zorder=0)
                ax.axvline(x0, color="black", ls="--", lw=1.2)
            ax.set_xlim(0, lags - 1)
            ax.grid(True, alpha=0.25, lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
            ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 4))
            if r == 0:
                ax.set_title(f"Latent dim {d}", pad=10)
            if r == n_rows - 1:
                ax.set_xlabel("Time lag")
            if d == 0:
                ax.set_ylabel(row_labels[r], fontweight="bold", fontsize=13)

    handles = [
        Line2D([0], [0], color=C_NORMAL, lw=2.5, label=lab_n),
        Line2D([0], [0], color=C_LAGGED, lw=2.5, label=lab_l),
    ]
    if supervision_t is not None:
        handles.append(Patch(facecolor="0.35", alpha=0.25, edgecolor="black",
                             ls="--", label=f"Supervised window (W = {supervision_t})"))
    
    leg = fig.legend(handles=handles, loc="upper center", ncol=len(handles),
                     bbox_to_anchor=(0.5, -0.01), fontsize=13,
                     columnspacing=2.5, handlelength=2.5)

    # grassetto sulla TV minore
    if bold_idx is not None:
        leg.get_texts()[bold_idx].set_fontweight("bold")

    if savepath is not None:
        fig.savefig(savepath, dpi=600, bbox_inches="tight", pad_inches=0.02)
    plt.show()
    del fig, axes

plot_temporal_embeddings_unified(
    H_normal_centered,
    H_lagged_centered,
    n_dims=4,
    n_samples=2,
    supervision_t=W_KEEPED_TIMESTEPS,
    label_normal="T-SHRED",            # adapt to your paper naming
    label_lagged="Lagged T-SHRED",
    seed=1,                            # change to get different random samples
    savepath="HYDRO_6_embeddings.png",
)

## Time plot

In [ ]:
def get_last_step_embeddings(model, dataset, device, batch_size=200):
    """Extract z_t from the last encoder state of every shifted sensor window."""
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    last_steps = []

    with torch.no_grad():
        for batch in loader:
            if MASKING_TYPE in ["mask_data", "mask_data_random"]:
                x, mask, _ = batch
                encoded = model.encoder(x.to(device), mask.to(device))
            else:
                x, _ = batch
                encoded = model.encoder(x.to(device))
            last_steps.append(encoded[-1].cpu())

    return torch.cat(last_steps, dim=0).numpy()


def total_variation_last_step(z, normalize=True, eps=1e-12):
    """Compute temporal variation of z_t, with z shaped (N, D)."""
    if normalize:
        z = z / (z.std(axis=0, keepdims=True) + eps)
    return np.linalg.norm(np.diff(z, axis=0), axis=1).sum()

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D



C_NORMAL = "#0072B2"   # blue
C_LAGGED = "#D55E00"   # vermilion

PAPER_RC = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.9,          # 9   * 1.1
    "axes.titlesize": 11,      # 10  * 1.1
    "axes.labelsize": 9.9,     # 9   * 1.1
    "xtick.labelsize": 8.8,    # 8   * 1.1
    "ytick.labelsize": 8.8,    # 8   * 1.1
    "legend.fontsize": 9.9,    # 9   * 1.1
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

z_normal = get_last_step_embeddings(normal_shred, test_dataset, device)
z_lagged = get_last_step_embeddings(lagged_shred, test_dataset, device)
print("z_normal shape:", z_normal.shape)
print("z_lagged shape:", z_lagged.shape)

def plot_last_step_embeddings_unified(
    z_normal, z_lagged, dims=(0, 2, 4, 6, 8), start=0, stop=None,
    label_normal="Last-step training", label_lagged="Multi-step training",
    normalize_tv=True, savepath=None, fig_width=10, panel_height=1.9,
):
    """Paper-style plot of mean-centered last-step latent trajectories (single row)."""
    if z_normal.shape != z_lagged.shape:
        raise ValueError(f"Expected matching latent shapes, got {z_normal.shape} and {z_lagged.shape}")
    stop = z_normal.shape[0] if stop is None else min(stop, z_normal.shape[0])
    if not 0 <= start < stop:
        raise ValueError(f"Invalid time interval [{start}, {stop})")
    dims = [d for d in dims if d < z_normal.shape[1]]
    if not dims:
        raise ValueError("None of the requested dims exist in the latent space")

    # TV computed on the plotted window (all dims), as before
    z_normal = z_normal[start:stop]
    z_lagged = z_lagged[start:stop]
    z_normal = z_normal - z_normal.mean(axis=0, keepdims=True)
    z_lagged = z_lagged - z_lagged.mean(axis=0, keepdims=True)
    t = np.arange(start, stop)

    tv_normal = total_variation_last_step(z_normal, normalize_tv)
    tv_lagged = total_variation_last_step(z_lagged, normalize_tv)
    change_pct = 100 * (tv_lagged - tv_normal) / max(tv_normal, 1e-12)

    ncols = len(dims)

    with plt.rc_context(PAPER_RC):
        fig, axes = plt.subplots(
            1, ncols, figsize=(fig_width * ncols / 2.5, panel_height + 0.6),
            squeeze=False, sharex=True, layout="constrained",
        )
        axes = axes.ravel()

        for i, (ax, d) in enumerate(zip(axes, dims)):
            ax.plot(t, z_lagged[:, d], color=C_LAGGED, lw=0.6, alpha=0.85, zorder=3,
                    rasterized=True)
            ax.plot(t, z_normal[:, d], color=C_NORMAL, lw=0.6, alpha=0.75, zorder=2,
                    rasterized=True)
            ax.set_title(rf"$z_{{t,{d}}}$", loc="left", pad=3)
            ax.set_xlim(t[0], t[-1])
            ax.margins(y=0.06)
            ax.grid(True, alpha=0.2, lw=0.4)
            ax.spines[["top", "right"]].set_visible(False)
            ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 3), useMathText=True)
            ax.set_xlabel("Test-set time step")

        axes[0].set_ylabel("Last-step Embedding")

        def tv_label(label, tv, is_best):
            # Bold only the number of the best (lowest) TV
            number = rf"$\mathbf{{{tv:.0f}}}$" if is_best else f"{tv:.0f}"
            return f"{label} (TV = {number})"

        best_is_normal = tv_normal <= tv_lagged
        handles = [
            Line2D([0], [0], color=C_NORMAL, lw=1.6,
                   label=tv_label(label_normal, tv_normal, best_is_normal)),
            Line2D([0], [0], color=C_LAGGED, lw=1.6,
                   label=tv_label(label_lagged, tv_lagged, not best_is_normal)),
        ]
        leg = fig.legend(handles=handles, loc="outside lower center", ncol=2,
                         frameon=False, columnspacing=2.5, handlelength=2.2)

        print(f"Last-step TV change (lagged vs normal): {change_pct:+.1f}%")

        if savepath is not None:
            fig.savefig(savepath, dpi=600)
            fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf")
        plt.show()
    return fig, axes


_ = plot_last_step_embeddings_unified(
    z_normal,
    z_lagged,
    dims=(2, 6, 10),
    label_normal="Last-step training",
    label_lagged="Multi-step training",
    savepath="HYDRO_6a_embeddings.png",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def get_last_step_embeddings(model, dataset, device, batch_size=200):
    """Extract z_t from the last encoder state of every shifted sensor window."""
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    last_steps = []

    with torch.no_grad():
        for batch in loader:
            if MASKING_TYPE in ["mask_data", "mask_data_random"]:
                x, mask, _ = batch
                encoded = model.encoder(x.to(device), mask.to(device))
            else:
                x, _ = batch
                encoded = model.encoder(x.to(device))
            last_steps.append(encoded[-1].cpu())

    return torch.cat(last_steps, dim=0).numpy()


def total_variation_last_step(z, normalize=True, eps=1e-12):
    """Temporal variation of z_t, with z shaped (N, D)."""
    if normalize:
        z = z / (z.std(axis=0, keepdims=True) + eps)
    return np.linalg.norm(np.diff(z, axis=0), axis=1).sum()


def total_variation(H, normalize=True, eps=1e-12):
    """H: (N, T, D). Optionally scale-normalize each sample by its RMS."""
    if normalize:
        H = H / (H.std(axis=(1, 2), keepdims=True) + eps)
    diff = H[:, 1:, :] - H[:, :-1, :]
    return np.linalg.norm(diff, axis=2).sum(axis=1)


def compute_tv_stats(H_normal, H_lagged, normalize=True):
    tv_n = total_variation(H_normal, normalize)
    tv_l = total_variation(H_lagged, normalize)
    return {
        "normal_mean": tv_n.mean(), "normal_std": tv_n.std(),
        "lagged_mean": tv_l.mean(), "lagged_std": tv_l.std(),
        "change_pct": 100 * (tv_l.mean() - tv_n.mean()) / tv_n.mean(),
    }


C_NORMAL = "#0072B2"   # blue
C_LAGGED = "#D55E00"   # vermilion

PAPER_RC = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.9,
    "axes.titlesize": 11,
    "axes.labelsize": 9.9,
    "xtick.labelsize": 8.8,
    "ytick.labelsize": 8.8,
    "legend.fontsize": 9.9,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

z_normal = get_last_step_embeddings(normal_shred, test_dataset, device)
z_lagged = get_last_step_embeddings(lagged_shred, test_dataset, device)
print("z_normal shape:", z_normal.shape)
print("z_lagged shape:", z_lagged.shape)


def _style_ax(ax):
    ax.margins(y=0.06)
    ax.grid(True, alpha=0.2, lw=0.4)
    ax.spines[["top", "right"]].set_visible(False)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 3), useMathText=True)


def _tv_number(is_best, *vals, pm=False):
    """Bold (mathtext) only the numbers of the best (lowest) TV."""
    txt = f"{vals[0]:.1f} \\pm {vals[1]:.1f}" if pm else f"{vals[0]:.0f}"
    return rf"$\mathbf{{{txt}}}$" if is_best else rf"${txt}$"

def plot_embeddings_merged(
    H_normal, H_lagged, z_normal, z_lagged,
    dims=(2, 6, 10, 14),
    supervision_t=None,
    label_normal="Last-step training",
    label_lagged="Multi-step training",
    normalize_tv=True, savepath=None,
    panel_width=3, panel_height=1.5,
):
    """(a) Mean ± 1 std of the temporal embeddings. (b) Last-step latent trajectories."""
    if z_normal.shape != z_lagged.shape:
        raise ValueError(f"Mismatching latent shapes: {z_normal.shape} vs {z_lagged.shape}")
    N, lags, hidden = H_normal.shape
    dims = [d for d in dims if d < hidden and d < z_normal.shape[1]]
    if not dims:
        raise ValueError("None of the requested dims exist in the latent space")
    ncols = len(dims)

    # ---------- TV (a) ----------
    s = compute_tv_stats(H_normal, H_lagged, normalize_tv)
    best_a_normal = s["normal_mean"] < s["lagged_mean"]
    print(f"(a) TV change (lagged vs normal): {s['change_pct']:+.1f}%")

    # ---------- TV (b) ----------
    zn = z_normal - z_normal.mean(axis=0, keepdims=True)
    zl = z_lagged - z_lagged.mean(axis=0, keepdims=True)
    tv_n = total_variation_last_step(zn, normalize_tv)
    tv_l = total_variation_last_step(zl, normalize_tv)
    best_b_normal = tv_n <= tv_l
    print(f"(b) Last-step TV change (lagged vs normal): "
          f"{100 * (tv_l - tv_n) / max(tv_n, 1e-12):+.1f}%")

    t_lag = np.arange(lags)
    t_test = np.arange(zn.shape[0])

    with plt.rc_context(PAPER_RC):
        # compact figure: each row = panel + xlabel + legend
        fig = plt.figure(figsize=(panel_width * ncols + 0.5, 2 * panel_height + 1.9),
                         layout="constrained")
        fig.get_layout_engine().set(h_pad=0.02, hspace=0.02, w_pad=0.02, wspace=0.03)
        sf_a, sf_b = fig.subfigures(2, 1, hspace=0.0)

        # ================= (a) mean ± 1 std =================
        axes_a = sf_a.subplots(1, ncols, sharex=True, squeeze=False)[0]
        for ax, d in zip(axes_a, dims):
            for H, c in [(H_normal, C_NORMAL), (H_lagged, C_LAGGED)]:
                mu, sd = H[:, :, d].mean(0), H[:, :, d].std(0)
                ax.plot(t_lag, mu, color=c, lw=0.9)
                ax.fill_between(t_lag, mu - sd, mu + sd, color=c, alpha=0.2, lw=0)
            if supervision_t is not None:
                x0 = lags - supervision_t
                ax.axvspan(x0, lags - 1, color="0.2", alpha=0.10, lw=0, zorder=0)
                ax.axvline(x0, color="black", ls="--", lw=0.8)
            ax.set_xlim(0, lags - 1)
            ax.set_title(rf"$\bar{{z}}_{{{d}}}$", loc="left", pad=3)   # same notation as (b)
            ax.set_xlabel("Time lag")
            _style_ax(ax)
        axes_a[0].set_ylabel(r"Embedding (mean $\pm$ 1 std)")

        handles_a = [
            Line2D([0], [0], color=C_NORMAL, lw=1.6,
                   label=f"{label_normal} (TV = "
                         f"{_tv_number(best_a_normal, s['normal_mean'], s['normal_std'], pm=True)})"),
            Line2D([0], [0], color=C_LAGGED, lw=1.6,
                   label=f"{label_lagged} (TV = "
                         f"{_tv_number(not best_a_normal, s['lagged_mean'], s['lagged_std'], pm=True)})"),
        ]
        if supervision_t is not None:
            handles_a.append(Patch(facecolor="0.35", alpha=0.25, edgecolor="black", ls="--",
                                   label=f"Supervised window (W = {supervision_t})"))
        sf_a.legend(handles=handles_a, loc="outside lower center", ncol=len(handles_a),
                    frameon=False, columnspacing=1.8, handlelength=1.8)
        sf_a.text(0.0, 1.0, "(a)", ha="left", va="top", fontweight="bold", fontsize=12)

        # ================= (b) last time step =================
        axes_b = sf_b.subplots(1, ncols, sharex=True, squeeze=False)[0]
        for ax, d in zip(axes_b, dims):
            ax.plot(t_test, zl[:, d], color=C_LAGGED, lw=0.6, alpha=0.85, zorder=3, rasterized=True)
            ax.plot(t_test, zn[:, d], color=C_NORMAL, lw=0.6, alpha=0.75, zorder=2, rasterized=True)
            ax.set_xlim(t_test[0], t_test[-1])
            ax.set_title(rf"$z_{{t,{d}}}$", loc="left", pad=3)
            ax.set_xlabel("Test-set time step")
            _style_ax(ax)
        axes_b[0].set_ylabel("Last-step embedding")

        handles_b = [
            Line2D([0], [0], color=C_NORMAL, lw=1.6,
                   label=f"{label_normal} (TV = {_tv_number(best_b_normal, tv_n)})"),
            Line2D([0], [0], color=C_LAGGED, lw=1.6,
                   label=f"{label_lagged} (TV = {_tv_number(not best_b_normal, tv_l)})"),
        ]
        sf_b.legend(handles=handles_b, loc="outside lower center", ncol=2,
                    frameon=False, columnspacing=1.8, handlelength=1.8)
        sf_b.text(0.0, 1.0, "(b)", ha="left", va="top", fontweight="bold", fontsize=12)

        if savepath is not None:
            fig.savefig(savepath, dpi=600)
            fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf")
        plt.show()
    return fig


_ = plot_embeddings_merged(
    H_normal_centered, H_lagged_centered,
    z_normal, z_lagged,
    dims=(2, 6, 10, 14),
    supervision_t=W_KEEPED_TIMESTEPS,
    label_normal="Last-step training",
    label_lagged="Multi-step training",
    savepath="HYDRO_6b_embeddings.png",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def get_last_step_embeddings(model, dataset, device, batch_size=200):
    """Extract z_t from the last encoder state of every shifted sensor window."""
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    last_steps = []

    with torch.no_grad():
        for batch in loader:
            if MASKING_TYPE in ["mask_data", "mask_data_random"]:
                x, mask, _ = batch
                encoded = model.encoder(x.to(device), mask.to(device))
            else:
                x, _ = batch
                encoded = model.encoder(x.to(device))
            last_steps.append(encoded[-1].cpu())

    return torch.cat(last_steps, dim=0).numpy()


def total_variation_last_step(z, normalize=True, eps=1e-12):
    """Temporal variation of z_t, with z shaped (N, D)."""
    if normalize:
        z = z / (z.std(axis=0, keepdims=True) + eps)
    return np.linalg.norm(np.diff(z, axis=0), axis=1).sum()


def total_variation(H, normalize=True, eps=1e-12):
    """H: (N, T, D). Optionally scale-normalize each sample by its RMS."""
    if normalize:
        H = H / (H.std(axis=(1, 2), keepdims=True) + eps)
    diff = H[:, 1:, :] - H[:, :-1, :]
    return np.linalg.norm(diff, axis=2).sum(axis=1)


def compute_tv_stats(H_normal, H_lagged, normalize=True):
    tv_n = total_variation(H_normal, normalize)
    tv_l = total_variation(H_lagged, normalize)
    return {
        "normal_mean": tv_n.mean(), "normal_std": tv_n.std(),
        "lagged_mean": tv_l.mean(), "lagged_std": tv_l.std(),
        "change_pct": 100 * (tv_l.mean() - tv_n.mean()) / tv_n.mean(),
    }


C_NORMAL = "#0072B2"   # blue
C_LAGGED = "#D55E00"   # vermilion

PAPER_RC = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.9,
    "axes.titlesize": 11,
    "axes.labelsize": 9.9,
    "xtick.labelsize": 8.8,
    "ytick.labelsize": 8.8,
    "legend.fontsize": 9.9,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

z_normal = get_last_step_embeddings(normal_shred, test_dataset, device)
z_lagged = get_last_step_embeddings(lagged_shred, test_dataset, device)
print("z_normal shape:", z_normal.shape)
print("z_lagged shape:", z_lagged.shape)


def _style_ax(ax):
    ax.margins(y=0.06)
    ax.grid(True, alpha=0.2, lw=0.4)
    ax.spines[["top", "right"]].set_visible(False)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 3), useMathText=True)


def _tv_number(is_best, *vals, pm=False):
    """Bold (mathtext) only the numbers of the best (lowest) TV."""
    txt = f"{vals[0]:.1f} \\pm {vals[1]:.1f}" if pm else f"{vals[0]:.0f}"
    return rf"$\mathbf{{{txt}}}$" if is_best else rf"${txt}$"

def plot_embeddings_merged(
    H_normal, H_lagged, z_normal=None, z_lagged=None,
    dims=(2, 6, 10, 14),
    supervision_t=None,
    label_normal="Last-step training",
    label_lagged="Multi-step training",
    normalize_tv=True, savepath=None,
    panel_width=3, panel_height=1.5,
):
    """Mean ± 1 std of the temporal embeddings (single row)."""
    N, lags, hidden = H_normal.shape
    dims = [d for d in dims if d < hidden]
    if not dims:
        raise ValueError("None of the requested dims exist in the latent space")
    ncols = len(dims)

    # ---------- TV ----------
    s = compute_tv_stats(H_normal, H_lagged, normalize_tv)
    best_normal = s["normal_mean"] < s["lagged_mean"]
    print(f"TV change (lagged vs normal): {s['change_pct']:+.1f}%")

    t_lag = np.arange(lags)

    with plt.rc_context(PAPER_RC):
        # single row: panel + xlabel + legend, no empty space
        fig = plt.figure(figsize=(panel_width * ncols + 0.5, panel_height + 1.0),
                         layout="constrained")
        fig.get_layout_engine().set(h_pad=0.02, hspace=0.02, w_pad=0.02, wspace=0.03)

        axes = fig.subplots(1, ncols, sharex=True, squeeze=False)[0]
        for ax, d in zip(axes, dims):
            for H, c in [(H_normal, C_NORMAL), (H_lagged, C_LAGGED)]:
                mu, sd = H[:, :, d].mean(0), H[:, :, d].std(0)
                ax.plot(t_lag, mu, color=c, lw=0.9)
                ax.fill_between(t_lag, mu - sd, mu + sd, color=c, alpha=0.2, lw=0)

            ax.set_xlim(0, lags - 1)
            ax.set_title(f"Latent dim {d + 1}", loc="left", pad=3, fontweight="normal")   # was \bar{z}_d
            ax.set_xlabel("Test time steps")                          # was "Time lag"
            _style_ax(ax)
        axes[0].set_ylabel(r"Embedding")

        handles = [
            Line2D([0], [0], color=C_NORMAL, lw=1.6,
                   label=f"{label_normal} (TV = "
                         f"{_tv_number(best_normal, s['normal_mean'], s['normal_std'], pm=True)})"),
            Line2D([0], [0], color=C_LAGGED, lw=1.6,
                   label=f"{label_lagged} (TV = "
                         f"{_tv_number(not best_normal, s['lagged_mean'], s['lagged_std'], pm=True)})"),
        ]
        
        fig.legend(handles=handles, loc="outside lower center", ncol=len(handles),
                   frameon=False, columnspacing=1.8, handlelength=1.8)

        if savepath is not None:
            fig.savefig(savepath, dpi=600, bbox_inches="tight", pad_inches=0.02)
            fig.savefig(savepath.rsplit(".", 1)[0] + ".pdf",
                        bbox_inches="tight", pad_inches=0.02)
        plt.show()
    return fig

_ = plot_embeddings_merged(
    H_normal_centered, H_lagged_centered,
    z_normal, z_lagged,
    dims=(0,1,2,3),
    supervision_t=W_KEEPED_TIMESTEPS,
    label_normal="Last-step training",
    label_lagged="Multi-step training",
    savepath="HYDRO_6_embeddings.png",
)